# Day 6: Session 6A - The Join Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6a_joining_data.html)

Date: 09/08/2026

In [1]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/openaq_goleta_measurments.csv'
goleta = pd.read_csv(url)

goleta['parameter'].value_counts()

parameter
pm25    734
o3      711
pm10    517
Name: count, dtype: int64

In [2]:
# filter pattern
o3 = goleta[goleta['parameter'] == 'o3'].copy()
pm25 = goleta[goleta['parameter'] == 'pm25'].copy()

print(o3.shape)
print(pm25.shape)

(711, 15)
(734, 15)


In [3]:
# filter to only the two columns that matter, 
# and give the measurement column a name 

o3 = o3[['datetimeLocal', 'value']]
o3 = o3.rename(columns={'value': 'o3_ppm'})

pm25 = pm25[['datetimeLocal', 'value']]
pm25 = pm25.rename(columns={'value': 'pm25_ugm3'})

o3.head()

,datetimeLocal,o3_ppm
0,2024-07-11T18:00:00-07:00,0.025
1,2024-07-11T19:00:00-07:00,0.028
2,2024-07-11T20:00:00-07:00,0.029
3,2024-07-11T21:00:00-07:00,0.027
4,2024-07-11T22:00:00-07:00,0.026


In [4]:
pm25.head()

,datetimeLocal,pm25_ugm3
1228,2024-07-11T18:00:00-07:00,3.0
1229,2024-07-11T19:00:00-07:00,8.0
1230,2024-07-11T20:00:00-07:00,6.0
1231,2024-07-11T21:00:00-07:00,4.0
1232,2024-07-11T22:00:00-07:00,9.0


### The join pattern
use pd.merge() function

In [5]:
paired = pd.merge(o3, pm25, on='datetimeLocal')

paired.head()

# pd.merge(left, right, on='key')
#           ↑     ↑         ↑
#        [the two tables]  [the column they have in common]

# how= inner/outer/left/right
# by default, how is set to inner (i.e. will do 
# inner join if we don't specify how to join)
# still, ALWAYS specify how you are merging (even if inner)



# the result below has one row for every hour that appears in both tables, 
# and the columns of both.
# pd.merge() keeps only the rows whose key appears in both tables

,datetimeLocal,o3_ppm,pm25_ugm3
0,2024-07-11T18:00:00-07:00,0.025,3.0
1,2024-07-11T19:00:00-07:00,0.028,8.0
2,2024-07-11T20:00:00-07:00,0.029,6.0
3,2024-07-11T21:00:00-07:00,0.027,4.0
4,2024-07-11T22:00:00-07:00,0.026,9.0


In [6]:
paired.shape

(704, 3)

In [7]:
inner = pd.merge(o3, pm25, on='datetimeLocal', how='inner')

inner.shape

(704, 3)

### Keeping everything on one side

Often you do not want the intersection. You have a table you care about, and a second table of extra information you would like to attach where it exists, without losing rows where it does not.

use how='left'



In [8]:
left = pd.merge(o3, pm25, on='datetimeLocal', how='left')

left.shape

(711, 3)

In [9]:
left.isnull().sum()

datetimeLocal    0
o3_ppm           0
pm25_ugm3        7
dtype: int64

In [10]:
right = pd.merge(o3, pm25, on='datetimeLocal', how='right')

print(right.shape)
print(right.isnull().sum())

(734, 3)
datetimeLocal     0
o3_ppm           30
pm25_ugm3         0
dtype: int64


In [11]:
outer = pd.merge(o3, pm25, on='datetimeLocal', how='outer')

print(outer.shape)
print(outer.isnull().sum())

(741, 3)
datetimeLocal     0
o3_ppm           30
pm25_ugm3         7
dtype: int64


### test your knowledge
Build a third table, pm10, the same way you built the other two tables: filter goleta to parameter == 'pm10', keep datetimeLocal and value, and rename value to pm10_ugm3. Then merge your PM2.5 table with it two ways, keeping pm25 on the left both times, once using how='inner' and once using how='left', and print both shapes. How many PM2.5 hours have no PM10 reading beside them?

In [12]:
pm10 = goleta[goleta['parameter'] == 'pm10'].copy()
pm10 = pm10[['datetimeLocal', 'value']]
pm10 = pm10.rename(columns={'value': 'pm10_ugm3'})

pm_left = pd.merge(pm25, pm10, on='datetimeLocal', how='left')
pm_inner = pd.merge(pm25, pm10, on='datetimeLocal', how='inner')

print(f"left: {pm_left.shape}")
print(f"inner: {pm_inner.shape}")

pm_left.isnull().sum()

left: (734, 3)
inner: (512, 3)


datetimeLocal      0
pm25_ugm3          0
pm10_ugm3        222
dtype: int64

### The rows you lose are not a random sample



In [13]:
missing_o3 = right[right['o3_ppm'].isnull()].copy()

missing_o3['datetimeLocal'].str[11:13].value_counts()

datetimeLocal
03    30
Name: count, dtype: int64

Every single one of the thirty PM2.5 hours with no ozone beside it is at 03:00. Not spread across the day: all of them, all thirty, at three in the morning, on thirty different days.

In [14]:
o3['datetimeLocal'].str[11:13].value_counts().sort_index()

datetimeLocal
00    31
01    30
02    30
04    31
05    31
06    31
07    31
08    31
09    31
10    31
11    31
12    31
13    31
14    31
15    31
16    31
17    31
18    31
19    31
20    31
21    31
22    31
23    31
Name: count, dtype: int64

### What a joined table is for



In [15]:
# the top-N pattern applied to the joined table:
paired.sort_values('pm25_ugm3', ascending=False).head(10)


,datetimeLocal,o3_ppm,pm25_ugm3
464,2024-08-01T01:00:00-07:00,0.011,22.0
142,2024-07-18T01:00:00-07:00,0.022,22.0
619,2024-08-08T01:00:00-07:00,0.025,21.0
673,2024-08-10T10:00:00-07:00,0.029,20.0
609,2024-08-07T15:00:00-07:00,0.043,19.0
585,2024-08-06T13:00:00-07:00,0.040,19.0
349,2024-07-27T01:00:00-07:00,0.008,17.0
293,2024-07-24T15:00:00-07:00,0.027,17.0
295,2024-07-24T17:00:00-07:00,0.026,16.0
586,2024-08-06T14:00:00-07:00,0.041,16.0


In [16]:
print(paired['o3_ppm'].mean())
print(paired.sort_values('pm25_ugm3', ascending=False).head(10)['o3_ppm'].mean())

0.022438920454545454
0.027200000000000002


In [17]:
# The mean of the ten smokiest hours is higher than the mean of every hour, 
# 0.029 ppm against 0.022

### When the key columns have different names



In [18]:
url = 'https://eds-217-essential-python.github.io/data/national_parks.csv'
parks = pd.read_csv(url)

parks = parks[parks['year'] != 'Total'].copy()

parks['region'].value_counts()

region
IM    5682
NE    3637
SE    3442
PW    3194
MW    2578
NC    1547
AK    1018
NT      76
Name: count, dtype: int64

In [19]:
regions = pd.DataFrame({
    'code': ['AK', 'IM', 'MW', 'NC', 'NE', 'PW', 'SE'],
    'region_name': ['Alaska', 'Intermountain', 'Midwest', 'National Capital',
                    'Northeast', 'Pacific West', 'Southeast'],
})

regions

,code,region_name
0,AK,Alaska
1,IM,Intermountain
2,MW,Midwest
3,NC,National Capital
4,NE,Northeast
5,PW,Pacific West
6,SE,Southeast


In [20]:
labelled = pd.merge(parks, regions, left_on='region', right_on='code')

labelled[['unit_name', 'region', 'region_name', 'year', 'visitors']].head()

,unit_name,region,region_name,year,visitors
0,Crater Lake National Park,PW,Pacific West,1904,1500.0
1,Lake Roosevelt National Recreation Area,PW,Pacific West,1941,0.0
2,Lewis and Clark National Historical Park,PW,Pacific West,1961,69000.0
3,Olympic National Park,PW,Pacific West,1935,2200.0
4,Santa Monica Mountains National Recreation Area,PW,Pacific West,1982,468144.0


In [21]:
### pd.merge(left, right, left_on='column_in_left', right_on='column_in_right')

In [22]:
print(parks.shape)
print(labelled.shape)

(21174, 12)
(21098, 14)


some rows didnt make it.  find them:

In [23]:
checked = pd.merge(parks, regions, left_on='region', right_on='code', how='left')

missing = checked[checked['region_name'].isnull()]

print(missing['region'].value_counts())
print(missing['unit_name'].unique())

region
NT    76
Name: count, dtype: int64
['Blue Ridge Parkway']


the NT rows did not make it